# 5.1 모델 없이 배운다: MC 예측 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter05_1_mc_prediction_lawnmower.ipynb)

책 본문: [5.1 모델 없이 배운다: MC 예측](https://smhanlab.com/book-ml/kor/ml2/chapter05.html)

잔디깎는 기(lawnmower) MDP에서 정책 \(\pi_0\)를 따라 에피소드를 샘플링하고,
**전이확률 0.8/0.2를 코드로 쓰지 않는** 첫방문 MC로 \(V^{\pi_0}\)를 추정합니다.
Chapter 4의 DP(선형방정식 풀기)로 구한 정확값과 비교하고,
"뒤에서부터 훑는" 마지막 방문 버그가 **편향**을 만든다는 것을 숫자로 확인합니다.


In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
import numpy as np

## 1. 잔디깎는 기 MDP (샘플러만 만든다)

3×3 격자. 네 모퉁이가 시작 상태(매 에피소드 균등 선택), 중앙 \((1,1)\)이 터미널.
정책 \(\pi_0\)의 **의도한** 다음 상태는 `NEXT`에 고정되어 있고,
0.8로 의도한 방향, 0.2로 정반대(미끄러짐; 벽이면 제자리)로 이동합니다.
보상은 이동마다 \(-0.1\), 터미널 진입 시 \(+1\). \(\gamma = 0.9\).

MC가 "모델을 모른다"는 말은 **추정기**가 이 0.8/0.2를 쓰지 않는다는 뜻이고,
환경(샘플러)은 원래 동역학대로 시뮬레이션합니다.


In [2]:
TERMINAL = (1, 1)
CORNERS = [(0, 0), (0, 2), (2, 0), (2, 2)]
# 정책 pi_0: 외곽을 순회하다 중앙(터미널)로 들어간다
NEXT = {
    (0, 0): (0, 1), (0, 1): (0, 2), (0, 2): (1, 2), (1, 2): TERMINAL,  # 위쪽 루프
    (2, 2): (2, 1), (2, 1): (2, 0), (2, 0): (1, 0), (1, 0): TERMINAL,  # 아래쪽 루프
}
P_INTENT, P_SLIP = 0.8, 0.2
GAMMA = 0.9

def step(s, rng):
    """정책 pi_0 하의 한 스텝. (다음 상태, 보상) 반환."""
    n = NEXT[s]
    if rng.random() < P_INTENT:
        nxt = n
    else:
        ox, oy = 2 * s[0] - n[0], 2 * s[1] - n[1]   # 정반대 방향
        nxt = (ox, oy) if 0 <= ox <= 2 and 0 <= oy <= 2 else s  # 벽이면 제자리
    r = 1.0 if nxt == TERMINAL else -0.1
    return nxt, r

def sample_episode(rng):
    """에피소드 1개를 샘플링: [(state, reward), ...] (터미널 진입 스텝까지)."""
    s = CORNERS[rng.integers(0, 4)]
    ep = []
    while True:
        nxt, r = step(s, rng)
        ep.append((s, r))
        s = nxt
        if s == TERMINAL:
            break
    return ep

rng = np.random.default_rng(0)
ep = sample_episode(rng)
print("예시 에피소드:", [(s, r) for s, r in ep])
print("길이:", len(ep))

예시 에피소드: [((2, 2), -0.1), ((2, 1), -0.1), ((2, 0), -0.1), ((1, 0), -0.1), ((1, 0), -0.1), ((1, 0), 1.0)]
길이: 6


## 2. 정확값: DP로 \((I - \gamma P)V = R\)를 푼다

전이확률을 **알고 있으므로** 8개 비터미널 상태에 대해 선형방정식을 직접 풉니다.
이 값이 MC가 "자기만의 방식으로" 도달해야 할 목표입니다.


In [3]:
states = list(NEXT.keys())   # 8개 비터미널 상태
idx = {st: i for i, st in enumerate(states)}
P = np.zeros((8, 8))
R = np.zeros(8)
for st in states:
    n = NEXT[st]
    d = (n[0] - st[0], n[1] - st[1])
    slip = (st[0] - d[0], st[1] - d[1])
    slip = slip if 0 <= slip[0] <= 2 and 0 <= slip[1] <= 2 else st
    if n != TERMINAL:      # 터미널 전이는 V=0이라 P에 질량 넣을 필요 없음
        P[idx[st], idx[n]] += P_INTENT
    if slip != TERMINAL:
        P[idx[st], idx[slip]] += P_SLIP
    R[idx[st]] = (P_INTENT * (1.0 if n == TERMINAL else -0.1)
                  + P_SLIP * (1.0 if slip == TERMINAL else -0.1))
V_exact = np.linalg.solve(np.eye(8) - GAMMA * P, R)
V_exact_map = {st: v for st, v in zip(states, V_exact)}
for st in states:
    print(f"V{st} = {V_exact_map[st]:+.3f}")
print("터미널 V(1,1) = 0.000")

V(0, 0) = +0.286
V(0, 1) = +0.465
V(0, 2) = +0.713
V(1, 2) = +0.951
V(2, 2) = +0.286
V(2, 1) = +0.465
V(2, 0) = +0.713
V(1, 0) = +0.951
터미널 V(1,1) = 0.000


## 3. 첫방문 MC vs 마지막 방문(버그) MC

20만 에피소드를 **한 번만** 샘플링하고(시드 42), 같은 에피소드 묶음으로
두 버전을 비교합니다. `mc_prediction`은 에피소드 기록만 쓰고,
0.8/0.2 같은 전이확률은 어디에도 쓰지 않습니다.

- **첫방문**: 리턴은 뒤에서부터 재귀 계산하되, 상태는 **앞에서부터** 훑어
  처음으로 만난 방문만 카운트.
- **마지막 방문(버그)**: 책 "자주 하는 실수"의 코드 그대로 — 뒤에서부터
  훑으면서 `if s not in visited`를 쓰면 사실상 마지막 방문을 세게 된다.


In [4]:
N_EP = 200_000
rng = np.random.default_rng(42)
episodes = [sample_episode(rng) for _ in range(N_EP)]

def mc_prediction(episodes, gamma, last_visit_bug=False):
    """첫방문(기본) 또는 마지막 방문(버그) MC. 전이확률은 전혀 쓰지 않는다."""
    tot, cnt = {}, {}
    for ep in episodes:
        G, rets = 0.0, []
        for s, r in reversed(ep):          # 리턴 G_t는 뒤에서부터 재귀 계산
            G = r + gamma * G
            rets.append(G)
        rets.reverse()
        visited = set()
        if last_visit_bug:
            # 책의 실수 코드: 뒤에서부터 훑으며 '아직 안 본' 상태를 카운트
            for (s, r), Gt in zip(reversed(ep), reversed(rets)):
                if s not in visited:
                    visited.add(s)
                    tot[s] = tot.get(s, 0.0) + Gt
                    cnt[s] = cnt.get(s, 0) + 1
        else:
            for (s, r), Gt in zip(ep, rets):  # 앞에서부터: 진짜 첫방문
                if s not in visited:
                    visited.add(s)
                    tot[s] = tot.get(s, 0.0) + Gt
                    cnt[s] = cnt.get(s, 0) + 1
    return {s: tot[s] / cnt[s] for s in tot}, cnt

V_first, cnt_first = mc_prediction(episodes, GAMMA)
V_last,  _         = mc_prediction(episodes, GAMMA, last_visit_bug=True)

print(f"{'상태':>6} {'정확값':>8} {'첫방문 MC':>10} {'오차':>8} {'마지막방문 MC':>14} {'방문횟수':>9}")
for st in states:
    e = V_exact_map[st]
    print(f"{str(st):>6} {e:>+9.3f} {V_first[st]:>+10.3f} {V_first[st]-e:>+8.3f} "
          f"{V_last[st]:>+14.3f} {cnt_first[st]:>9,}")


    상태      정확값     첫방문 MC       오차       마지막방문 MC      방문횟수
(0, 0)    +0.286     +0.287   +0.000         +0.388    50,155
(0, 1)    +0.465     +0.465   +0.000         +0.542    50,155
(0, 2)    +0.713     +0.713   -0.000         +0.756   100,333
(1, 2)    +0.951     +0.951   -0.000         +1.000   100,333
(2, 2)    +0.286     +0.286   -0.000         +0.387    49,650
(2, 1)    +0.465     +0.465   -0.001         +0.541    49,650
(2, 0)    +0.713     +0.712   -0.001         +0.756    99,667
(1, 0)    +0.951     +0.951   -0.000         +1.000    99,667


## 4. 수렴: 누적 평균과 리턴 분포

좌: \((0,0)\), \((0,2)\)의 첫방문 리턴 **누적 평균**이 정확값에 수렴해가는
모습(x축 로그 스케일). 우: \((0,0)\)의 첫방문 리턴 분포 — 단일 리턴의
분산이 크지만, 평균은 \(\sigma/\sqrt{N}\)로 진정됩니다.


In [5]:
IMG = "/home/smhan/book-ml/kor/src/images"
TARGETS = [(0, 0), (0, 2)]
fv = {t: [] for t in TARGETS}
for ep in episodes:
    G, rets = 0.0, []
    for s, r in reversed(ep):
        G = r + GAMMA * G
        rets.append(G)
    rets.reverse()
    seen = set()
    for (s, r), Gt in zip(ep, rets):
        if s in TARGETS and s not in seen:
            seen.add(s)
            fv[s].append(Gt)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
for t, color in zip(TARGETS, ["tab:blue", "tab:orange"]):
    mean = np.cumsum(fv[t]) / np.arange(1, len(fv[t]) + 1)
    ax1.plot(np.arange(1, len(fv[t]) + 1), mean, color=color, lw=1.2, label=str(t))
    ax1.axhline(V_exact_map[t], color=color, ls="--", lw=0.8)
ax1.set_xscale("log")
ax1.set_xlabel("First-visit count N")
ax1.set_ylabel("Cumulative mean return")
ax1.set_title("Cumulative mean first-visit returns for (0,0) and (0,2)")
ax1.legend()
ax1.grid(alpha=0.3)

rv = fv[(0, 0)]
ax2.hist(rv, bins=60, color="tab:blue", alpha=0.75)
ax2.axvline(V_exact_map[(0, 0)], color="k", ls="--", lw=1,
            label=f"Exact {V_exact_map[(0,0)]:.3f}")
ax2.axvline(np.mean(rv), color="r", ls=":", lw=1.2,
            label=f"Mean {np.mean(rv):.3f} (std {np.std(rv, ddof=1):.3f})")
ax2.set_xlabel("First-visit return")
ax2.set_title("First-visit return distribution for (0,0)")
ax2.legend()
ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch05_1_mc_convergence.svg", bbox_inches="tight")
plt.show()
print("(0,0): 방문", len(fv[(0,0)]), "회, 평균 %.3f, 표준편차 %.3f" % (np.mean(rv), np.std(rv, ddof=1)))

(0,0): 방문 50155 회, 평균 0.287, 표준편차 0.191


## 정리

- **DP**는 전이확률을 써서 \((I-\gamma P)V = R\)를 풀고, **첫방문 MC**는
  에피소드만 평균내도 같은 값에 도달한다(오차 ~0.001 수준).
- **마지막 방문(뒤에서부터 훑는) 버전**은 모든 상태에서 정확값보다 높은
  **편향된** 추정치를 준다 — 에피소드를 늘려도 사라지지 않는 편향이다.
- 수렴 그래프에서 "단일 리턴의 잡음"과 "평균의 진정(\(\sigma/\sqrt{N}\))"을
  나란히 확인할 수 있다.
